In [1]:
# ============================================================
# CMAPSS FD001 — Classification (Healthy / Warning / Failure)
# - Train set: train_FD001.txt
# - Compute RUL per engine (unit)
# - Labels:
#     Failure  : RUL <= 50
#     Warning  : 50 < RUL <= 125
#     Healthy  : RUL > 125
# - Split by ENGINE (GroupShuffleSplit) to avoid leakage
# - Drop constant columns based on TRAIN ONLY (avoid leakage)
# - StandardScaler only for Logistic/KNN/LinearSVM; Trees unscaled
# - Metrics: Accuracy + Macro F1 + Failure Recall + Classification Report + Confusion Matrix
# - Plots: Class Distribution, Confusion Matrix, RF Feature Importance
# - Saves figures + CSV
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# -------------------- CONFIG --------------------
PATH = "/kaggle/input/nasa-cmaps/CMaps/train_FD001.txt"

RUL_FAILURE = 50
RUL_WARNING = 125

TEST_SIZE = 0.2
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Label mapping (IMPORTANT: keep consistent everywhere)
# 0: Healthy, 1: Warning, 2: Failure
LABEL_NAMES = {0: "Healthy", 1: "Warning", 2: "Failure"}
LABEL_ORDER = [0, 1, 2]

OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# -------------------- LOAD + PREP --------------------
def load_fd001(path: str) -> pd.DataFrame:
    col_names = (
        ["unit_id", "cycle", "setting1", "setting2", "setting3"]
        + [f"s{i}" for i in range(1, 22)]
    )
    df = pd.read_csv(path, sep=r"\s+", header=None, names=col_names)
    return df


def add_rul(df: pd.DataFrame) -> pd.DataFrame:
    max_cycle = df.groupby("unit_id")["cycle"].max()
    out = df.copy()
    out["RUL"] = out["unit_id"].map(max_cycle) - out["cycle"]
    return out


def add_labels(df: pd.DataFrame) -> pd.DataFrame:
    rul = df["RUL"].to_numpy()
    y = np.zeros_like(rul, dtype=int)  # default Healthy=0
    y[(rul > RUL_FAILURE) & (rul <= RUL_WARNING)] = 1  # Warning=1
    y[rul <= RUL_FAILURE] = 2  # Failure=2
    out = df.copy()
    out["label"] = y
    return out


# -------------------- PLOTS --------------------
def save_class_distribution(y: pd.Series, out_path: str):
    counts = y.value_counts().reindex(LABEL_ORDER, fill_value=0)
    plt.figure(figsize=(6, 4))
    plt.bar([LABEL_NAMES[i] for i in LABEL_ORDER], counts.values)
    plt.title("Class Distribution (All Samples)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_confusion_matrix(y_true, y_pred, out_path: str, title: str):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)

    plt.figure(figsize=(7, 5))
    plt.imshow(cm)
    plt.title(title)
    plt.xticks(range(len(LABEL_ORDER)), [LABEL_NAMES[i] for i in LABEL_ORDER])
    plt.yticks(range(len(LABEL_ORDER)), [LABEL_NAMES[i] for i in LABEL_ORDER])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_rf_feature_importance(rf, feature_names, out_path: str, top_n=15):
    importances = rf.feature_importances_
    idx = np.argsort(importances)[::-1][:top_n]

    plt.figure(figsize=(12, 6))
    plt.bar([feature_names[i] for i in idx], importances[idx])
    plt.title(f"Random Forest Feature Importance (Top {top_n})")
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# -------------------- EVAL --------------------
def evaluate(model_name: str, y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)

    rep_txt = classification_report(
        y_true, y_pred,
        labels=LABEL_ORDER,
        target_names=[LABEL_NAMES[i] for i in LABEL_ORDER],
        zero_division=0
    )
    rep_dict = classification_report(
        y_true, y_pred,
        labels=LABEL_ORDER,
        target_names=[LABEL_NAMES[i] for i in LABEL_ORDER],
        output_dict=True,
        zero_division=0
    )

    macro_f1 = rep_dict["macro avg"]["f1-score"]
    failure_recall = rep_dict["Failure"]["recall"]

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print(f"Accuracy      : {acc:.4f}")
    print(f"Macro F1      : {macro_f1:.4f}")
    print(f"Failure Recall: {failure_recall:.4f}")
    print(rep_txt)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "MacroF1": macro_f1,
        "FailureRecall": failure_recall,
        "y_pred": y_pred
    }


# -------------------- MAIN --------------------
def main():
    print("--- Step 1: Loading & Preprocessing Dataset (CMAPSS FD001) ---")
    df = load_fd001(PATH)
    df = add_rul(df)
    df = add_labels(df)

    # features + target + groups
    features = ["setting1", "setting2", "setting3"] + [f"s{i}" for i in range(1, 22)]
    X = df[features].copy()
    y = df["label"].astype(int).copy()
    groups = df["unit_id"].astype(int).copy()

    # Plot (all samples)
    save_class_distribution(y, os.path.join(OUT_DIR, "fig_class_distribution.png"))

    # Split by engine
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

    # Drop constant columns based on TRAIN ONLY
    nunique = X_train.nunique(dropna=False)
    keep_cols = nunique[nunique > 1].index.tolist()
    X_train = X_train[keep_cols]
    X_test = X_test[keep_cols]

    # Scale for linear/distance models only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\n--- Step 2: Training Predictive Models ---")
    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE, solver="saga"
        ),
        "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
        "Linear SVM": LinearSVC(dual=False, max_iter=5000),
        "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }

    results = []
    preds = {}

    for name, model in models.items():
        start = time.time()

        if name in ["Decision Tree", "Random Forest"]:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        else:
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

        elapsed = time.time() - start
        preds[name] = y_pred

        # Evaluate with extra metrics
        r = evaluate(name, y_test, y_pred)
        r["Time(s)"] = round(elapsed, 2)
        results.append({k: v for k, v in r.items() if k != "y_pred"})

    print("\n--- Step 3: Summary ---")
    df_results = pd.DataFrame(results).sort_values(
        ["MacroF1", "FailureRecall", "Accuracy"], ascending=False
    )
    print(df_results.to_string(index=False))
    df_results.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"), index=False)

    best_name = df_results.iloc[0]["Model"]
    best_pred = preds[best_name]

    # Save confusion matrix for best model
    save_confusion_matrix(
        y_test, best_pred,
        os.path.join(OUT_DIR, "fig_confusion_matrix.png"),
        title=f"Confusion Matrix (Best Model: {best_name})"
    )

    # Save RF feature importance (train RF on unscaled)
    rf = models["Random Forest"]
    rf.fit(X_train, y_train)
    save_rf_feature_importance(
        rf, keep_cols, os.path.join(OUT_DIR, "fig_rf_feature_importance.png"), top_n=15
    )

    print("\nSaved files in ./outputs:")
    print(" - fig_class_distribution.png")
    print(" - fig_confusion_matrix.png")
    print(" - fig_rf_feature_importance.png")
    print(" - model_comparison.csv")
    print("\nBest model:", best_name)


if __name__ == "__main__":
    main()


--- Step 1: Loading & Preprocessing Dataset (CMAPSS FD001) ---

--- Step 2: Training Predictive Models ---

MODEL: Logistic Regression
Accuracy      : 0.7452
Macro F1      : 0.7563
Failure Recall: 0.8980
              precision    recall  f1-score   support

     Healthy       0.76      0.75      0.76      1550
     Warning       0.66      0.63      0.65      1500
     Failure       0.84      0.90      0.87      1020

    accuracy                           0.75      4070
   macro avg       0.75      0.76      0.76      4070
weighted avg       0.74      0.75      0.74      4070


MODEL: KNN (k=5)
Accuracy      : 0.7039
Macro F1      : 0.7178
Failure Recall: 0.8176
              precision    recall  f1-score   support

     Healthy       0.68      0.77      0.72      1550
     Warning       0.61      0.55      0.58      1500
     Failure       0.88      0.82      0.85      1020

    accuracy                           0.70      4070
   macro avg       0.72      0.72      0.72      4070
we